# Заняття 10 · ДЕМО 4 (самостійна робота)
## Компресія і точність моделі

Досі кодек цікавив нас як джерело затримки. Для CV-інженера важливіше інше: **модель бачить не кадр сенсора, а результат кодека**. Перевіряємо це на сцені, де легко порахувати точність: небо з хмарами, шум сенсора і 16 дрібних темних цілей розміром 3–8 px — дрони вдалині.

Детектор навмисно іграшковий (віднімання розмитого фону, поріг, зв’язні компоненти): нам потрібна не найкраща модель, а чесне порівняння однієї й тієї самої моделі на сирих і стиснутих кадрах. З вашою нейромережею дослід повторюється так само — замініть `detect_small` на свій детектор і recall на mAP.

| Крок | Що робимо |
|---|---|
| §1 | Сцена і детектор: recall на сирих кадрах — це інференс **до** енкодера |
| §2 | Сходинки бітрейту через справжній x264: recall за розміром цілі |
| §3 | Калібрування на стиснутому потоці: іграшковий аналог донавчання з аугментацією кодеком |

Мітка на слайдах: слайд 22 «Компресія і точність моделі» (розділ 03 «Енкодер і zero-copy»); на лекції не запускається — це «ще ноутбук для самостійної роботи» зі слайда 3. GStreamer живе в `streamlab.py` / `gstworker.py`, стиль графіків — у `panels.py`.

**Версія для самостійної роботи.** Клітинки з `TODO` (`raise NotImplementedError`) дописуєте самі; решта коду — як на лекції.

In [ ]:
# streamlab запускає GStreamer в окремому процесі (gstworker.py), тож у ноутбуці gi не імпортується
import json
import os

import numpy as np
import matplotlib.pyplot as plt
import streamlab as sl
from panels import PAL, set_style

set_style()
print(sl.gst_version(), '| appsink leaky-type (1.28+):', sl.appsink_has_leaky_type())

sc = sl.SkyScene(640, 360, 30)
BITRATES = [1600, 800, 400, 200, 100]            # кбіт/с для 640×360 при 30 fps
# threads=1: багатопотоковий x264 стискає той самий кадр щоразу трохи інакше, і recall дрібних цілей стрибає
# між прогонами; порівнюємо сходинки, тому енкодер має бути детермінованим
ENC = 'tune=zerolatency speed-preset=superfast bitrate={kb} key-int-max=30 threads=1'
SIZES = sorted(set(int(s) for s in sc.small_targets(0)[:, 2]))


def evaluate(frames, thr=12.0):
    """Recall, хибні спрацювання на кадр і recall за розміром цілі для списку (номер, кадр)."""
    tp = fp = fn = 0
    by = {s: [0, 0] for s in SIZES}
    for i, im in frames:
        gt = sc.small_targets(i)
        a, b, c, ts = sl.match(gt, sl.detect_small(im, sc.code_h, thr=thr))
        tp, fp, fn = tp + a, fp + b, fn + c
        for s in gt[:, 2]:
            by[int(s)][1] += 1
        for s in ts:
            by[int(s)][0] += 1
    return dict(recall=tp / (tp + fn), fp=fp / len(frames), by_size={s: v[0] / v[1] for s, v in by.items()})

### 1. Сцена і детектор на сирих кадрах
Це випадок, коли інференс іде на борту **до** енкодера: модель бачить кадр сенсора.

In [ ]:
raw = [(i, sc.frame(i)) for i in range(30, 180, 10)]
res = {'raw': evaluate(raw)}
print(f"сирі кадри: recall {res['raw']['recall']:.0%}, хибних спрацювань {res['raw']['fp']:.2f} на кадр")
fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(raw[2][1]); ax.axis('off')
for x, y, s in sc.small_targets(raw[2][0]):
    ax.add_patch(plt.Circle((x, y), s + 4, fill=False, color=PAL['or_'], lw=1.2))
ax.set_title('Сцена: 16 цілей 3–8 px (обведено), шум сенсора σ = 4')
plt.tight_layout(); plt.show()

### 2. Сходинки бітрейту
Кожна сходинка — справжній прогін x264 (`tune=zerolatency`) і декодер. Беремо кожен десятий кадр після першої секунди.

In [ ]:
frames = {}
for kb in BITRATES:
    # TODO: проженіть сцену через x264 (ENC з бітрейтом kb кбіт/с) і збережіть кожен 10-й декодований кадр (keep_every)
    raise NotImplementedError("TODO: проженіть сцену через x264 (ENC з бітрейтом kb кбіт/с) і збережіть кожен 10-й декодований кадр (keep_every)")
    frames[kb] = [(r['idx'], r['img']) for r in rec if 'img' in r and r['idx'] >= 30]
    res[kb] = evaluate(frames[kb])
    print(f"{kb:5d} кбіт/с: recall {res[kb]['recall']:.0%}, хибних {res[kb]['fp']:.2f} на кадр")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={'width_ratios': [1.3, 1]})
cols = [PAL['re'], PAL['or_'], PAL['bl'], PAL['te']]
for s, c in zip(SIZES, cols):
    ax[0].plot(BITRATES, [res[kb]['by_size'][s] * 100 for kb in BITRATES], 'o-', color=c, label=f'{s} px')
    ax[0].axhline(res['raw']['by_size'][s] * 100, color=c, lw=0.6, ls=':')
ax[0].set_xscale('log'); ax[0].set_xticks(BITRATES); ax[0].set_xticklabels(BITRATES)
ax[0].set_xlabel('бітрейт, кбіт/с (640×360, 30 fps)'); ax[0].set_ylabel('recall, %'); ax[0].set_ylim(0, 105)
ax[0].set_title('Recall за розміром цілі'); ax[0].legend(title='ціль')
i0, im0 = frames[BITRATES[-1]][2]
x, y, s = min(sc.small_targets(i0), key=lambda t: t[2])
cut = lambda im: im[int(y) - 20:int(y) + 20, int(x) - 30:int(x) + 30]
ax[1].imshow(np.hstack([cut(sc.frame(i0)), np.full((40, 4, 3), 255, np.uint8), cut(im0)]), interpolation='nearest')
ax[1].set_title(f'Ціль {int(s)} px: сирий кадр | {BITRATES[-1]} кбіт/с'); ax[1].axis('off')
plt.tight_layout(); plt.show()

Малі цілі зникають першими: блокове квантування згладжує контраст у кілька пікселів раніше, ніж око помітить різницю на всій картинці. На найнижчій сходинці з’являються ще й хибні спрацювання на артефактах.

### 3. Калібрування на стиснутому потоці
Якщо модель мусить працювати на землі, по стиснутому потоку, її треба навчати й калібрувати на кадрах того самого кодека і бітрейту. Іграшковий аналог донавчання — підібрати поріг детектора на кадрах 200 кбіт/с замість сирих.

In [ ]:
KB = 200
rows = []
for thr in (12, 11, 10, 9, 8):
    r = evaluate(frames[KB], thr=thr)
    rows.append((thr, r['recall'], r['fp']))
    print(f'поріг {thr:2d}: recall {r["recall"]:.0%}, хибних {r["fp"]:.2f} на кадр')
# TODO: виберіть поріг із найвищим recall за умови не більше 0.1 хибного спрацювання на кадр
raise NotImplementedError("TODO: виберіть поріг із найвищим recall за умови не більше 0.1 хибного спрацювання на кадр")
raw_at_best = evaluate(raw, thr=best[0])
print(f'\nпоріг для сирих кадрів — 12; для {KB} кбіт/с найкращий {best[0]}: recall {rows[0][1]:.0%} → {best[1]:.0%}')
print(f'той самий поріг {best[0]} на сирих кадрах: хибних {raw_at_best["fp"]:.2f} на кадр — оптимум зсувається разом із кодеком')
os.makedirs('outputs', exist_ok=True)                 # тека в .gitignore
with open('outputs/demo4_results.json', 'w') as f:
    json.dump({'bitrates': BITRATES, 'sizes': SIZES, 'raw': res['raw'],
               'ladder': {str(kb): res[kb] for kb in BITRATES},
               'calib': {'kb': KB, 'thr': best[0], 'recall_before': rows[0][1], 'recall_after': best[1]}}, f, indent=1)

### Висновки для слайда
* **Інференс до енкодера, на борту** — модель бачить кадр сенсора, і стиснення на точність не впливає. Це головний аргумент за CV на борту, разом із затримкою.
* **Якщо модель працює по стиснутому потоку** (аналітика на станції, запис) — навчайте й валідуйте її на кадрах того самого кодека і бітрейту: аугментація кодеком у тренуванні, окремий поріг для кожної сходинки ABR.
* **Валідуйте на кожній сходинці ABR**, а не лише на сирих кадрах: під РЕБ контролер опускає бітрейт саме тоді, коли модель найпотрібніша.
* Малі цілі — перші жертви: піксельний бюджет із Заняття 9 треба рахувати **після** кодека, а не лише після ресайзу.